# Environmental Footprints of Housing
This code calculates environmental footprints of housing under a given set of energy use, energy technology, housing size of different building types, and transport use in relation to location and income.

For the analysis of current housing in NL the following datasets are needed:
- Set of assumptions: CBS microdata output
- Material use per building types
- Material use per solar PV capacity
- Material use per heat pump
- District heat source distribution
- Fuel use per pkm
- Infrastructural material use per pkm

In [1]:
# data processing modules
import pandas as pd
import numpy as np

# data visualisation modules
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.express as px
import seaborn as sns

In [2]:
def multiply_subcategories(df_detail, df_aggregate):
    """ 
    Multiplies a detailed dataframe with an aggregated dataframe

    Returns a detailed dataframe in which each each row is multiplied with the corresponding row in the aggregated dataframe
    Index level 0 of the detailed dataframe must match the index of the aggregated dataframe.
    
    Args:
        df_detail: multi-indexed dataframe of length a*b and width c
        df_aggregate: dataframe of length a and width c
    
    Returns: 
        pandas.DataFrame: Detailed dataframe of length a*b and width c in which each each row is multiplied with the corresponding row in the aggregated dataframe
        """
    df_new = pd.DataFrame(index=df_detail.index, columns=df_detail.columns)
    for i in df_aggregate.index:
        df_new.update(df_detail.xs(i, axis=0, level=0, drop_level=False).mul(df_aggregate.loc[i], axis=1))
        
    return df_new


## Foreground System: Calculate and Harmonize Direct Material, Fuel & Electricity Use

### Set of Assumptions
- configure assumptions
- import the CBS file

In [2]:
# Configurations
scenario = 'current'
material_buildings_source = 'Oorschot-MPG' #alternative 'Oorschot' contains a different grouping (semi-detached and detached together, instead of row and semi-detached)
material_solarpv_source = 'ecoinvent'
material_heatpump_source = 'ecoinvent'
material_pkminfra_source = None
fuel_pkm_source = None
fuel_districtheating_source = None

# File import
if scenario == "current":
    use = pd.read_excel(
        "data_input/output/income typology construction density distance privacy 2019.xlsx",
        index_col=[0,1,2,3,4],
        header=[0],
    )

In [3]:
use

inc_decile  \
inc_lab             typology_lab              const_lab density_lab distance_train_lab               
decile 1 (poorest)  detached (vrijstaand)     pre 1945  very urban  nearest quintile             1   
                                                                    2nd quintile                 1   
                                                                    3nd quintile                 1   
                                                                    4nd quintile                 1   
                                                                    furthest quintile            1   
...                                                                                            ...   
decile 10 (richest) multiple fam (meergezins) 2006-2022 rural       nearest quintile            10   
                                                                    2nd quintile                10   
                                                                    3nd quintile                10   
                                                                    4nd quintile                10   
                                                                    furthest quintile           10   

                                                                                        typology4_2019  \
inc_lab             typology_lab              const_lab density_lab distance_train_lab                   
decile 1 (poorest)  detached (vrijstaand)     pre 1945  very urban  nearest quintile                 1   
                                                                    2nd quintile                     1   
                                                                    3nd quintile                     1   
                                                                    4nd quintile                     1   
                                                                    furthest quintile                1   
...                                                                                                ...   
decile 10 (richest) multiple fam (meergezins) 2006-2022 rural       nearest quintile                 4   
                                                                    2nd quintile                     4   
                                                                    3nd quintile                     4   
                                                                    4nd quintile                     4   
                                                                    furthest quintile                4   

                                                                                        const_cat2019  \
inc_lab             typology_lab              const_lab density_lab distance_train_lab                  
decile 1 (poorest)  detached (vrijstaand)     pre 1945  very urban  nearest quintile                1   
                                                                    2nd quintile                    1   
                                                                    3nd quintile                    1   
                                                                    4nd quintile                    1   
                                                                    furthest quintile               1   
...                                                                                               ...   
decile 10 (richest) multiple fam (meergezins) 2006-2022 rural       nearest quintile                6   
                                                                    2nd quintile                    6   
                                                                    3nd quintile                    6   
                                                                    4nd quintile                    6   
                                                                    furthest quintile               6   

                

In [4]:
use['size2019_w1']*use['n_hhd']/use['n_hhdmem2019']

inc_lab              typology_lab               const_lab  density_lab  distance_train_lab
decile 1 (poorest)   detached (vrijstaand)      pre 1945   very urban   nearest quintile       96.396156
                                                                        2nd quintile           99.423205
                                                                        3nd quintile           81.229786
                                                                        4nd quintile          172.199080
                                                                        furthest quintile            NaN
                                                                                                 ...    
decile 10 (richest)  multiple fam (meergezins)  2006-2022  rural        nearest quintile       72.131577
                                                                        2nd quintile           72.454547
                                                                     

In [5]:
use['size2019_w1']/use['mean_hhd']

inc_lab              typology_lab               const_lab  density_lab  distance_train_lab
decile 1 (poorest)   detached (vrijstaand)      pre 1945   very urban   nearest quintile       96.396156
                                                                        2nd quintile           99.423206
                                                                        3nd quintile           81.229782
                                                                        4nd quintile          172.199084
                                                                        furthest quintile            NaN
                                                                                                 ...    
decile 10 (richest)  multiple fam (meergezins)  2006-2022  rural        nearest quintile       72.131577
                                                                        2nd quintile           72.454547
                                                                     

In [6]:
(use['size2019_w1']*use['n_hhd']) / (use['size2019pp_w1']*use['n_hhdmem2019'])

inc_lab              typology_lab               const_lab  density_lab  distance_train_lab
decile 1 (poorest)   detached (vrijstaand)      pre 1945   very urban   nearest quintile      0.917509
                                                                        2nd quintile          0.910995
                                                                        3nd quintile          0.832690
                                                                        4nd quintile          1.332152
                                                                        furthest quintile          NaN
                                                                                                ...   
decile 10 (richest)  multiple fam (meergezins)  2006-2022  rural        nearest quintile      0.999307
                                                                        2nd quintile          0.784545
                                                                        3nd quintile 

In [14]:
use['size2019pp_w1']

inc_lab              typology_lab               const_lab  density_lab  distance_train_lab
decile 1 (poorest)   detached (vrijstaand)      pre 1945   very urban   nearest quintile      105.062950
                                                                        2nd quintile          109.136978
                                                                        3nd quintile           97.551064
                                                                        4nd quintile          129.263885
                                                                        furthest quintile            NaN
                                                                                                 ...    
decile 10 (richest)  multiple fam (meergezins)  2006-2022  rural        nearest quintile       72.181580
                                                                        2nd quintile           92.352272
                                                                     

In [5]:
use.index.levels[1]

Index(['detached (vrijstaand)', 'multiple fam (meergezins)',
       'row house (tussenwoning)', 'semi-detached (2-1 kap) / corner (hoek)'],
      dtype='object', name='typology_lab')

### Building Construction
- translate floor space to BONSAI material categories
- include material use for maintenance
- normalise impact to annual using building lifetime

In [10]:
# Loading the material intensities per floor area
if material_buildings_source == "Oorschot-MPG": #kg/m2
    mi_buildings = pd.read_excel(
        "data_input/MI_Buildings_Oorschot2023MetabolicMPG.xlsx",
        sheet_name="Building_MI",
        index_col=[0, 1],
        header=[0, 1], skiprows=3
    ).drop("2018-2050", axis=0, level=1)
    mi_buildings.index.names = ["Building type", "Cohort"]
    mi_buildings.columns.names = ["BONSAI", "Material"]
    mi_buildings.fillna(0, inplace=True)
elif material_buildings_source == "Oorschot": #kg/m2
    mi_buildings = pd.read_excel(
        "data_input/MI_Buildings_Oorschot2023.xlsx",
        sheet_name="mi_pre2018",
        index_col=[0, 1],
        header=[0, 1],
    ).droplevel(1, axis=1)
    mi_buildings.index.names = ["Building type", "Cohort"]
    mi_buildings.columns.name = "Material"

    #converting the material intensities to BONSAI material categories
    mi_buildings_insulation_share = pd.read_excel(
            "data_input/MI_Buildings_Oorschot2023.xlsx",
            sheet_name="Share_of_insulation_materials",
            index_col=[0, 1],
            header=[0],
            skipfooter=3,
        ).drop("Unnamed: 8", axis=1)
    for i in mi_buildings_insulation_share.columns:
        mi_buildings[i] = (
            mi_buildings["Insulation"] * mi_buildings_insulation_share[i].iloc[0]
            )

    mi_buildings = (
            mi_buildings
            .T.groupby(level=0)
            .sum()
            .T.sort_index()
            .replace(0, np.nan)
            .dropna(how="all", axis=1)
            .drop("Insulation", axis=1)
        )
    # matching material names
    replacement_map = {'Bitumen':'Bitumen', #soon
                        'Brick':'Ceramics', #BRIK,
                        'Clay brick':'Ceramics', #BRIK
                        'Ceramics':'Ceramics', #BRIK (bricks, tiles, construction products)
                        'Sand-lime brick':'Ceramics', #BRIK
                        'Sand':'Sand', #SDCL
                        'Concrete':'Concrete', #CMNT (only lime,cement and plast, no mix)
                        'Mortal':'Mortar', #CMNT
                        'Gypsum':'Mortar', #CMNT (plaster of gypsum)
                        'Aluminum':'Aluminium', #ALUM
                        'Copper':'Copper', #COPP
                        'Other metals':'Other metals', #ONFM
                        'Electronics':'Electronics', #ELMA
                        'Glass':'Glass', #GLAS 
                        'Glass wool':'Glass', #GLAS (includes explicitely glass wool)
                        'Stone wool':'Mineral wool', #ONMM (includes explicitely mineral wool)
                        'Iron & steel':'Steel', #STEL or FABM
                        'Steel':'Steel', #STEL or FABM
                        'Other biobased materials':'Wood and straw', #WOOD
                        'Wood':'Wood and straw', #WOOD
                        'Other construction materials':'Other construction material', #ONMM, could also be STON, SDCL or CMNT
                        'Other':'Adhesives and paint', #could be CHEM, PLAS, PAPE, ELMA
                        'Paper':'Paper', #PAPE 
                        'Expanded polystyrene':'Plastic', #RUBP 
                        'Extruded polystyrene':'Plastic', #RUBP
                        'Plastic':'Plastic', #RUBP
                        'Plastics':'Plastic', #RUBP
                        'Polyurethanes':'Plastic', #RUBP
                        'Adhesives & paint':'Adhesives and paint', #CHEM (other chemical products, for glues) or PLAS (basic plastics) 
                        'Stone':'Stone', #STON
                        }

    mi_buildings = mi_buildings.stack().reset_index()
    mi_buildings['Material_group'] = mi_buildings['Material'].map(replacement_map)
    mi_buildings = mi_buildings.drop('Material', axis=1).groupby(['Building type', 'Cohort', 'Material_group']).sum().unstack().droplevel(0, axis=1)

    #selecting only residential buildings
    mi_buildings.drop(['industry', 'office', 'services', 'shop'], axis=0, level=0, inplace=True)




In [13]:
# visualisation of the assumed material intensities
""" the trend shows a shift from wood to brick and concrete"""
m_plot = mi_buildings.copy()
m_plot = m_plot.rename(index={'pre 1945':'1000-1945'})
if material_buildings_source == "Oorschot-MPG":
    m_plot = m_plot.groupby(axis=1, level=0).sum()
m_plot.index = [str(i) + ' ' + str(j) for i, j in m_plot.index]
m_plot = m_plot.stack().reset_index()
m_plot.columns = ['building', 'material', 'value']
fig = px.bar(m_plot, x='value', y='building', color='material', labels={'value':'material intensity per floor area [kg/m2]'}, template='simple_white', 
             color_discrete_sequence= px.colors.qualitative.Prism) #antique, vivid, prism, T10
fig.update_layout(width=1100, height=500, yaxis = dict(autorange="reversed"))    
fig.write_image('visualisations/mi_buildings'+material_buildings_source+'_grouped.svg')
fig.show()

In [ ]:
#calculating the material contained in an average dwelling per category
mi_buildings.index = mi_buildings.index.rename(['typology_lab', 'const_lab']) # kg/m2 ufa
use_m2 = use['size2019_w1'].reorder_levels(['typology_lab', 'const_lab', 'inc_lab', "density_lab", "distance_train_lab"]) #m2 ufa / dwelling
use_m2 = pd.concat([use_m2] * len(mi_buildings.columns), axis=1)
use_m2.columns = mi_buildings.columns
material_buildings = use_m2.multiply(mi_buildings.reindex(use_m2.index, method="ffill"), level=[0,1]) #kg / dwelling

In [ ]:
material_buildings

BONSAI                                                                                            M_ALUM  \
Material                                                                               Aluminium (kg/m2)   
typology_lab              const_lab inc_lab             density_lab distance_train_lab                     
detached (vrijstaand)     pre 1945  decile 1 (poorest)  very urban  nearest quintile           37.397599   
                                                                    2nd quintile               34.863122   
                                                                    3nd quintile               35.015216   
                                                                    4nd quintile               60.125312   
                                                                    furthest quintile                NaN   
...                                                                                                  ...   
multiple fam (meergezins) 2006-2022 decile 10 (richest) rural       nearest quintile           27.983977   
                                                                    2nd quintile               35.136595   
                                                                    3nd quintile               30.750168   
                                                                    4nd quintile               32.707155   
                                                                    furthest quintile          30.908631   

BONSAI                                                                                           M_BRIK  \
Material                                                                               Baksteen (kg/m2)   
typology_lab              const_lab inc_lab             density_lab distance_train_lab                    
detached (vrijstaand)     pre 1945  decile 1 (poorest)  very urban  nearest quintile       22374.215642   
                                                                    2nd quintile           20857.890085   
                                                                    3nd quintile           20948.884458   
                                                                    4nd quintile           35971.739346   
                                                                    furthest quintile               NaN   
...                                                                                                 ...   
multiple fam (meergezins) 2006-2022 decile 10 (richest) rural       nearest quintile       16742.238589   
                                                                    2nd quintile           21021.502873   
                                                                    3nd quintile           18397.193989   
                                                                    4nd quintile           19568.018949   
                                                                    furthest quintile      18491.999200   

BONSAI                                                                                 M_CMNT + M_SDCL  \
Material                                                                                 Beton (kg/m2)   
typology_lab              const_lab inc_lab             density_lab distance_train_lab                   
detached (vrijstaand)     pre 1945  decile 1 (poorest)  very urban  nearest quintile     252714.877339   
                                                                    2nd quintile         235588.108151   
                                                                    3nd quintile         236615.881915   
                                                                    4nd quintile         406297.759978   
                                                                    furthest quintile              NaN   
...                                                                                                ...   
multiple fam (meergezins) 2006-202

### Building Use Phase
- translate district heating to BONSAI fuel categories
- translate solar PV and heat pumps to BONSAI material categories
- harmonise labels for electricity and gas use with BONSAI categories

### Mobility
- translate pkm to BONSAI fuel categories
- translate pkm to infrastructure demand to BONSAI material categories
- normalise cars to annual cars using car lifetime
- harmonise labels for cars with BONSAI categories

In [50]:
# File import
mobility = pd.read_excel(
        "data_input/output/income density distance privacy odin 2019.xlsx",
        index_col=[0,1,2],
        header=[0],
    )

use_pkm = mobility['car_trips']*mobility['dist_carpp'] #pkm per year
use_pkm.name = 'car_pkm'
use = use.join(use_pkm, on=['inc_lab','density_lab', 'distance_train_lab'])

use_pkm = mobility['transit_trips']*mobility['dist_transit'] #pkm per year
use_pkm.name = 'transit_pkm'
use = use.join(use_pkm, on=['inc_lab','density_lab', 'distance_train_lab'])

use_pkm = mobility['walk_trips']*mobility['dist_walk'] #pkm per year
use_pkm.name = 'walk_pkm'
use = use.join(use_pkm, on=['inc_lab','density_lab', 'distance_train_lab'])

use_pkm = mobility['cycle_trips']*mobility['dist_cycle'] #pkm per year
use_pkm.name = 'cycle_pkm'
use = use.join(use_pkm, on=['inc_lab','density_lab', 'distance_train_lab'])

use = use.join(mobility['car_hhd'], on=['inc_lab','density_lab', 'distance_train_lab'])

## Background System: Environmental Footprints

### Environmental impact factors
- import BONSAI multipliers

In [ ]:
mult_bonsai = pd.read_csv('data_input/240605_BONSAI_ghg_for_all_iluc.csv', index_col=[0,1,2,3], header=[0], skipfooter=9) # per tonne

prod_codes = material_buildings.columns.levels[0].to_list()
prod_codes.append('M_CMNT')
mult_bonsai = mult_bonsai[mult_bonsai.index.get_level_values(0).isin(prod_codes)] 
mult_bonsai = mult_bonsai.droplevel([1,2,3])
mult_bonsai = mult_bonsai[['NL']] #select only NL market

concrete_cement_pc = 0.35 #based on M20 and M25 concrete grade
mult_bonsai.loc['M_CMNT + M_SDCL'] = mult_bonsai.loc[['M_CMNT']].squeeze()*concrete_cement_pc + mult_bonsai.loc[['M_SDCL']].squeeze()*(1-concrete_cement_pc)

/var/folders/__/twfrdq8n75n5f0118_f882700000gn/T/ipykernel_18720/3140546246.py:1: ParserWarning:

Falling back to the 'python' engine because the 'c' engine does not support skipfooter; you can avoid this warning by specifying engine='python'.



In [49]:
mult_bonsai

,NL
Exio prod code,
M_STON,0.001885
M_SDCL,0.000796
M_WOOD,0.889511
M_PAPE,2.433502
M_BITU,1.785329
M_CHEM,1.058851
M_RUBP,3.791940
M_GLAS,1.294315
M_BRIK,0.175436


In [50]:
material_buildings_grouped = material_buildings.groupby(axis=1, level=0).sum()/1000 #t/dwelling
ghg_material_buildings = material_buildings_grouped.multiply(mult_bonsai.iloc[:,0], axis=1) #t CO2eq. / dwelling

In [ ]:
ghg_material_buildings.sum(1) #use['size2019_w1']

typology_lab               const_lab  inc_lab              density_lab  distance_train_lab
detached (vrijstaand)      pre 1945   decile 1 (poorest)   very urban   nearest quintile      135.728709
                                                                        2nd quintile          126.530223
                                                                        3nd quintile          127.082222
                                                                        4nd quintile          218.215369
                                                                        furthest quintile       0.000000
                                                                                                 ...    
multiple fam (meergezins)  2006-2022  decile 10 (richest)  rural        nearest quintile      101.563445
                                                                        2nd quintile          127.522747
                                                                     

# CODE ARCHIVE

In [ ]:
# Configurations
mi_buildings_source = "Oorschot"  # Oorschot or Yang
# Load data
if mi_buildings_source == "Oorschot":
    mi_buildings = pd.read_excel(
        "data_input/MI_Buildings_Oorschot2023.xlsx",
        sheet_name="MI_2018",
        index_col=[0, 1],
        header=[0, 1],
    ).droplevel(1, axis=1)
    mi_buildings.index.names = ["Building type", "Cohort"]
    mi_buildings.columns.name = "Material"

    mi_buildings_2018 = pd.read_excel(
        "data_input/MI_Buildings_Oorschot2023.xlsx",
        sheet_name="MI_2018-2050",
        index_col=[0],
        header=[0, 1],
        skipfooter=20,
    ).droplevel(1, axis=1)
    mi_buildings_2018["Cohort"] = "2018"
    mi_buildings_2018.reset_index(inplace=True)
    mi_buildings_2018["index"] = [i.lower() for i in mi_buildings_2018["index"]]
    mi_buildings_2018.set_index(["index", "Cohort"], inplace=True)
    mi_buildings_2018.index.names = ["Building type", "Cohort"]
    mi_buildings_2018.columns.name = "Material"

    mi_buildings_insulation_share = pd.read_excel(
        "data_input/MI_Buildings_Oorschot2023.xlsx",
        sheet_name="Share_of_insulation_materials",
        index_col=[0, 1],
        header=[0],
        skipfooter=3,
    ).drop("Unnamed: 8", axis=1)
    for i in mi_buildings_insulation_share.columns:
        mi_buildings[i] = (
            mi_buildings["Insulation"] * mi_buildings_insulation_share[i].iloc[0]
        )
        mi_buildings_2018[i] = (
            mi_buildings_2018["Insulation"] * mi_buildings_insulation_share[i].iloc[1]
        )

    mi_buildings = (
        pd.concat([mi_buildings, mi_buildings_2018], axis=1)
        .T.groupby(level=0)
        .sum()
        .T.sort_index()
        .replace(0, np.nan)
        .dropna(how="all", axis=1)
        .drop("Insulation", axis=1)
    )

# matching material names
replacement_map = {'Bitumen':'Bitumen', #soon
                    'Brick':'Ceramics', #BRIK,
                    'Clay brick':'Ceramics', #BRIK
                    'Ceramics':'Ceramics', #BRIK (bricks, tiles, construction products)
                    'Sand-lime brick':'Ceramics', #BRIK
                    'Sand':'Sand', #SDCL
                    'Concrete':'Concrete', #CMNT (only lime,cement and plast, no mix)
                    'Mortal':'Mortar', #CMNT
                    'Gypsum':'Mortar', #CMNT (plaster of gypsum)
                    'Aluminum':'Aluminium', #ALUM
                    'Copper':'Copper', #COPP
                    'Other metals':'Other metals', #ONFM
                    'Electronics':'Electronics', #ELMA
                    'Glass':'Glass', #GLAS 
                    'Glass wool':'Glass', #GLAS (includes explicitely glass wool)
                    'Stone wool':'Mineral wool', #ONMM (includes explicitely mineral wool)
                    'Iron & steel':'Steel', #STEL or FABM
                    'Steel':'Steel', #STEL or FABM
                    'Other biobased materials':'Wood and straw', #WOOD
                    'Wood':'Wood and straw', #WOOD
                    'Other construction materials':'Other construction material', #ONMM, could also be STON, SDCL or CMNT
                    'Other':'Adhesives and paint', #could be CHEM, PLAS, PAPE, ELMA
                    'Paper':'Paper', #PAPE 
                    'Expanded polystyrene':'Plastic', #RUBP 
                    'Extruded polystyrene':'Plastic', #RUBP
                    'Plastic':'Plastic', #RUBP
                    'Plastics':'Plastic', #RUBP
                    'Polyurethanes':'Plastic', #RUBP
                    'Adhesives & paint':'Adhesives and paint', #CHEM (other chemical products, for glues) or PLAS (basic plastics) 
                    'Stone':'Stone', #STON
                    }

mi_buildings = mi_buildings.stack().reset_index()
mi_buildings['Material_group'] = mi_buildings['Material'].map(replacement_map)
mi_buildings.replace('<1945','1944', inplace=True)
mi_buildings = mi_buildings.drop('Material', axis=1).groupby(['Building type', 'Cohort', 'Material_group']).sum().unstack().droplevel(0, axis=1)

#selecting only residential buildings
mi_buildings.drop(['industry', 'office', 'services', 'shop'], axis=0, level=0, inplace=True)

TypeError: can't multiply sequence by non-int of type 'float'

In [ ]:
# Configurations
ei_heating_source = "TABULA"  # Oorschot or Yang
# Load data
if ei_heating_source == "TABULA":
    ei_heating = pd.read_excel(
        "data_input/EI_Heating_HotWater_TABULA2017.xlsx",
        sheet_name="0_EnergyDemandIntensity",
        index_col=[0],
        header=[0],
    ).drop('Default Heating & Hot Water', axis=1)

In [ ]:
m_plot = ei_heating.copy()

m_plot = m_plot.stack().reset_index()
m_plot.columns = ['building', 'function', 'value']
fig = px.bar(m_plot, x='value', y='building', color='function', labels={'value':'energy intensity per floor area [kWh/m2]'}, template='simple_white', 
             color_discrete_sequence= px.colors.qualitative.Prism) #antique, vivid, prism, T10
fig.update_layout(width=500, height=1200, yaxis = dict(autorange="reversed"))    
fig.write_image('visualisations/ei_heating'+ei_heating_source+'.svg')
fig.show()

In [ ]:
# Configurations
mi_buildings_source = "Yang"  # Oorschot or Yang
# Load data
if mi_buildings_source == "Yang":
    mi_buildings = pd.read_excel(
        "data_input/MI_Buildings_Yang2022.xlsx",
        sheet_name='0_MaterialIntensity',
        index_col=[0, 1],
        header=[0, 1],
    ).droplevel(0, axis=0).droplevel(1, axis=1)
    mi_buildings.index.names = ["Building type"]
    mi_buildings.columns.name = "Material"

    # matching material names
    replacement_map = {'bitumen adhesive compound, hot':'Bitumen', #soon
                        'clay brick':'Ceramics', #BRIK
                        'ceramic tile':'Ceramics', #BRIK (bricks, tiles, construction products)
                        'sand':'Sand', #SDCL
                        'concrete, normal':'Concrete', #CMNT (only lime,cement and plast, no mix)
                        'fibre-reinforced concrete':'Concrete', #CMNT (only lime,cement and plast, no mix)
                        'cement mortar':'Mortar', #CMNT
                        'gypsum, mineral':'Mortar', #CMNT (plaster of gypsum)
                        'aluminium, wrought alloy':'Aluminium', #ALUM ##
                        'copper':'Copper', #COPP
                        'zinc':'Zinc', #LZNC
                        'flat glass, coated':'Glass', #GLAS 
                        'glass wool mat':'Glass', #GLAS (includes explicitely glass wool)
                        'cast iron':'Steel', #STEL or FABM
                        'wire drawing, steel':'Steel', #STEL or FABM
                        'plywood, for indoor use':'Wood and straw', #WOOD
                        'sawnwood, hardwood, raw':'Wood and straw', #WOOD
                        'sawnwood, softwood, raw':'Wood and straw', #WOOD
                        'fibreboard, soft':'Wood and straw', #WOOD
                        'polystyrene, expandable':'Plastic', #RUBP 
                        'polystyrene, extruded':'Plastic', #RUBP
                        'extrusion, plastic pipes':'Plastic', #RUBP
                        'polyurethane, flexible foam':'Plastic', #RUBP
                        'gravel, crushed':'Stone', #STON
                        'argon, liquid':'Argon', #CHEM
                        }

    mi_buildings = mi_buildings.stack().reset_index()
    mi_buildings['Material_group'] = mi_buildings['Material'].map(replacement_map)
    mi_buildings = mi_buildings.drop('Material', axis=1).groupby(['Building type', 'Material_group']).sum().unstack().droplevel(0, axis=1)

    nzeb = mi_buildings.loc[(mi_buildings.index.str.contains('_NZEB')) & ~(mi_buildings.index.str.contains('new'))]
    nzeb.index = [i[:-5] for i in nzeb.index]
    nzeb = nzeb + mi_buildings.loc[~(mi_buildings.index.str.contains('_NZEB')) & ~(mi_buildings.index.str.contains('new'))]
    nzeb.index = [i+'_NZEB' for i in nzeb.index]
    mi_buildings.update(nzeb)

In [ ]:
m_plot = mi_buildings[~mi_buildings.duplicated()]
m_plot = m_plot.stack().reset_index()
m_plot.columns = ['building', 'material', 'value']
fig = px.bar(m_plot, x='value', y='building', color='material', labels={'value':'material intensity per floor area [kg/m2]', 'sector':'Input sector'}, template='simple_white', 
             color_discrete_sequence= px.colors.qualitative.Prism) #antique, vivid, prism, T10
fig.update_layout(width=1100, height=700, yaxis = dict(autorange="reversed"))    
fig.write_image('visualisations/mi_buildings'+mi_buildings_source+'.svg')
fig.show()